# Exploratory Data Analysis (EDA) - FairCraft AI (Etsy High-Performance Dataset)

L'objectif de ce document est de visualiser les interactions entre nos variables de marché réelles (prix, avis, catégories) et les composants synthétiques avancés que nous avons construits pour maximiser les performances de l'IA (comme `popularity_index`, et la log-distanciation du prix).

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

sns.set_theme(style="whitegrid", palette="pastel")
plt.rcParams['figure.figsize'] = (14, 6)

import warnings
warnings.filterwarnings('ignore')

## 1. Inspection des données Nettoyées

In [2]:
DATA_PATH = '../../data/processed/etsy_clean.csv'
if os.path.exists(DATA_PATH):
    df = pd.read_csv(DATA_PATH)
    print(f"Dimensions du Dataset: {df.shape}")
    display(df.head())
else:
    print("Fichier etsy_clean.csv introuvable. Exécutez data_pipeline.py d'abord.")

Fichier etsy_clean.csv introuvable. Exécutez data_pipeline.py d'abord.


## 2. Analyse des Valeurs Manquantes
Nous validons ici que le pipeline ETL fonctionne bien de bout en bout et que nous n'injectons aucun Null mortel dans notre `TransformedTargetRegressor`.

In [3]:
if 'df' in locals():
    print("Valeurs Nulles existantes :")
    print(df.isnull().sum()[df.isnull().sum() > 0])
    
    plt.figure(figsize=(10,3))
    sns.heatmap(df.isnull(), cbar=False, cmap='viridis')
    plt.title('Heatmap des chaleurs manquantes (Devrait être vide)')
    plt.show()

## 3. Distribution des Prix (Target Variable)
La visualisation de `price_numeric` montre habituellement l'hétéroscédasticité naturelle de l'artisanat. On voit l'impact du transform log (`price_log`) sur lequel XGBoost sur-performe.

In [4]:
if 'df' in locals():
    fig, ax = plt.subplots(1, 2, figsize=(16, 5))
    
    sns.histplot(df['price_numeric'], bins=40, kde=True, ax=ax[0], color='dodgerblue')
    ax[0].set_title('Distribution Brute du Prix ($)')
    
    sns.histplot(df['price_log'], bins=40, kde=True, ax=ax[1], color='darkorange')
    ax[1].set_title('Transformation Logarithmique du Prix')
    plt.show()

## 4. Comparatif Prix par Catégorie Artisanale

In [5]:
if 'df' in locals():
    plt.figure(figsize=(12, 6))
    sns.boxplot(y='category', x='price_numeric', data=df, palette='Set2')
    plt.title('Outliers et Baselines par sous-groupe (Etsy)')
    plt.show()

## 5. Matrice de Corrélation
Vérification de la puissance prédictive linéaire avant de laisser XGBoost modéliser les complexités non-linéaires.

In [6]:
if 'df' in locals():
    plt.figure(figsize=(10, 8))
    numeric_df = df.select_dtypes(include=[np.number])
    sns.heatmap(numeric_df.corr(), annot=True, cmap='coolwarm', fmt='.2f', vmin=-1, vmax=1)
    plt.title('Relations linéaires entre features synthétiques')
    plt.show()